# Road PCI Intelligence — Stage 3: RDD2022ES Manifest Builder (Parallel)

This notebook builds the object detection manifest for **RDD2022ES** (`juusos/rdd2022es`) using multi-threaded I/O:
- **Group Splitting**: Strips `xmirror_` prefix to ensure each original image and its horizontally-flipped counterpart share the exact same split (80% train / 10% val / 10% test).
- **Parallel Dimension Extraction**: Uses `concurrent.futures.ThreadPoolExecutor(max_workers=16)` to read image header dimensions across all 25,600 files concurrently.
- **Why Image Dimensions Are Recorded**:
  While YOLO bounding box coordinates (`class_id x_center y_center w h`) are normalized in $[0, 1]$, RDD2022ES contains disparate camera resolutions across collection regions (e.g., Japan: 600x600, 720x720; India: 1080x1080; Czech: 600x600). Storing `(width, height)` in the manifest is essential for un-normalizing bounding boxes, pixel-space evaluation, aspect-ratio analysis, and multi-scale anchor generation.


In [ ]:
# ==============================================================================
# 1. PARALLEL RDD2022ES MANIFEST BUILDER (16 WORKERS)
# ==============================================================================
import os
import sys
import time
import random
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from PIL import Image

t0_rdd = time.time()
print('=' * 80)
print('STARTING PARALLEL RDD2022ES MANIFEST GENERATION')
print('=' * 80)

# Resolve dataset root
candidates = [
    Path('/kaggle/input/datasets/juusos/rdd2022es'),
    Path('/kaggle/input/rdd2022es'),
    Path('data/rdd2022es')
]
dataset_root = None
for c in candidates:
    if c.exists() and c.is_dir():
        dataset_root = c.resolve()
        break

if dataset_root is None:
    k_input = Path('/kaggle/input')
    if k_input.exists():
        for d in k_input.rglob('rdd2022es'):
            if d.is_dir():
                dataset_root = d.resolve()
                break

if dataset_root is None:
    raise FileNotFoundError("Could not find rdd2022es dataset root.")

print(f"Dataset root found: {dataset_root}")

annotated_dir = dataset_root / 'combined_annotatedv2'
if not annotated_dir.exists():
    dirs = list(dataset_root.rglob('combined_annotatedv2'))
    annotated_dir = dirs[0] if dirs else dataset_root

img_files = sorted(list(annotated_dir.glob('*.jpg')))
print(f"Discovered {len(img_files):,} images in {annotated_dir.name}")

def get_rdd_group_id(filename: str) -> str:
    stem = Path(filename).stem
    if stem.startswith('xmirror_'):
        return stem[len('xmirror_'):]
    return stem

# Group images by base scene
group_to_imgs = defaultdict(list)
for img_p in img_files:
    gid = get_rdd_group_id(img_p.name)
    group_to_imgs[gid].append(img_p)

unique_groups = sorted(list(group_to_imgs.keys()))
rng = random.Random(42)
shuffled_groups = unique_groups.copy()
rng.shuffle(shuffled_groups)

n_total = len(shuffled_groups)
n_train = int(round(n_total * 0.80))
n_val = int(round(n_total * 0.10))

train_groups = set(shuffled_groups[:n_train])
val_groups = set(shuffled_groups[n_train:n_train + n_val])
test_groups = set(shuffled_groups[n_train + n_val:])

print(f"Unique Groups: {n_total:,} | Train: {len(train_groups):,} | Val: {len(val_groups):,} | Test: {len(test_groups):,}")

# Flatten items
work_items = []
for gid, imgs in group_to_imgs.items():
    if gid in train_groups:
        split = 'train'
    elif gid in val_groups:
        split = 'val'
    else:
        split = 'test'

    for img_p in imgs:
        txt_p = img_p.with_suffix('.txt')
        rel_img = img_p.relative_to(dataset_root).as_posix()
        rel_txt = txt_p.relative_to(dataset_root).as_posix() if txt_p.exists() else ""
        work_items.append((img_p, rel_img, rel_txt, gid, split))

def process_single_image(item):
    img_p, rel_img, rel_txt, gid, split = item
    try:
        with Image.open(img_p) as img:
            w, h = img.size
    except Exception:
        w, h = 640, 640
        
    return {
        'source': 'rdd2022es',
        'image_path': rel_img,
        'mask_path_or_label_path': rel_txt,
        'group_id': gid,
        'split': split,
        'width': w,
        'height': h,
        'labels_crack': False,
        'labels_pothole': False
    }

print(f"Reading image dimensions in parallel with 16 workers...")
t_dim_start = time.time()
with ThreadPoolExecutor(max_workers=16) as executor:
    records = list(executor.map(process_single_image, work_items))

t_dim_elapsed = time.time() - t_dim_start
print(f"Completed dimension extraction for {len(records):,} images in {t_dim_elapsed:.2f}s ({len(records)/t_dim_elapsed:.1f} img/s)")

df_rdd = pd.DataFrame(records)
out_dir = Path('manifests')
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / 'rdd2022es_manifest.csv'
df_rdd.to_csv(out_csv, index=False)

t_total_rdd = time.time() - t0_rdd
print(f"Saved {out_csv} ({len(df_rdd):,} records)")
print(f"[RDD BUILD TIME] Total elapsed time: {t_total_rdd:.2f}s")


In [ ]:
# ==============================================================================
# 2. VERIFY RDD2022ES INTEGRITY & SPLIT PROPERTIES
# ==============================================================================
print('=' * 80)
print('VALIDATING RDD2022ES MANIFEST SPLITS & PAIRING')
print('=' * 80)

# Check leakage
leakage = df_rdd.groupby('group_id')['split'].nunique()
n_leaked = (leakage > 1).sum()
print(f"Group ID leakage across splits: {n_leaked} (Expected: 0)")
assert n_leaked == 0, "Data leakage detected in RDD manifest!"

# Check mirrored pair count
img_per_group = df_rdd.groupby('group_id')['image_path'].count()
all_two = (img_per_group == 2).all()
print(f"All groups contain exactly 2 paired images (original + mirrored): {all_two}")
assert all_two, "Incomplete mirror pairs found!"

# Check split distribution
print("\nSplit distribution:")
for sp, cnt in df_rdd['split'].value_counts().items():
    print(f"  • {sp:5s}: {cnt:,} images ({cnt/len(df_rdd)*100:.1f}%)")

print("\n[SUCCESS] RDD2022ES manifest verified.")
